# 01 - Baseline: 20M-parameter GPT, 50M tokens, biggest batch that fits

**Goal (assignment part 1).** Train a ~20M-parameter LLM on 50M tokens with an ordinary
(non-reversible) transformer. Find the biggest batch size that fits in GPU memory, fix it, and
record final loss, speed (tokens/s) and peak memory. Notebooks 02 and 03 reuse this batch size.

Set the runtime to a **GPU** (Runtime > Change runtime type). Edit `REPO_URL` in the next cell.

In [ ]:
SMOKE = False   # True = tiny CPU version of this whole notebook (used to test the code; not for results)
REPO_URL = "https://github.com/pratik-2025/Asisgnment13_Reversible_LLM_20M"   # <- edit: only needed on Colab
USE_DRIVE = True   # True = keep data + results on Google Drive so they survive a Colab disconnect

import os, sys, copy, json, subprocess
if os.path.exists("../revlm"):
    os.chdir("..")                                   # opened from the notebooks/ folder
elif not os.path.exists("revlm"):
    assert "YOUR-USERNAME" not in REPO_URL, "edit REPO_URL (top of this cell) to point at your GitHub repo"
    subprocess.run(["git", "clone", REPO_URL, "repo"], check=True)   # fresh Colab VM
    os.chdir("repo")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "-q", "install", "tokenizers", "datasets"], check=False)

import torch
from IPython.display import Image, display
from revlm import selftest, experiment as ex
from revlm.bench import bench_steps, scaling_table
from revlm.data import prepare_data, Data
from revlm.report import write_summary, plot_scaling

S = ex.settings(SMOKE)
DEV = ex.device()
RESULTS = "results_smoke" if SMOKE else "results"
DATA_DIR = "data_smoke" if SMOKE else "data"
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    RESULTS = "/content/drive/MyDrive/reversible-llm-20m/" + RESULTS
    DATA_DIR = "/content/drive/MyDrive/reversible-llm-20m/" + DATA_DIR
print("device:", DEV, torch.cuda.get_device_name() if DEV == "cuda" else "(no GPU!)" if not SMOKE else "(smoke test)")
if DEV != "cuda" and not SMOKE:
    print("WARNING: no GPU. In Colab use Runtime > Change runtime type > GPU.")

## 0. Sanity check

In [ ]:
# 30-second correctness check on THIS machine/GPU before spending an hour of compute.
assert selftest.main(DEV), "self-test failed - do not trust any numbers below"

## 1. Data

In [ ]:
# Downloads TinyStories, trains an 8k BPE tokenizer, writes exactly 50M training tokens (~3-6 min, cached).
prepare_data(DATA_DIR, vocab_size=8192, **S["data"])
data = Data(DATA_DIR, S["model"].ctx, seed=1337)
print(f"train tokens available: {len(data.train):,}   validation tokens: {len(data.val):,}   sequences: {data.n_seq:,}")

In [ ]:
def mk(mode, **over):
    """20M-parameter model config in the given mode (same weights/shape for every mode)."""
    mc = copy.deepcopy(S["model"]); mc.mode = mode
    for k, v in over.items():
        setattr(mc, k, v)
    return mc

from revlm.model import GPT
print(f"parameters: {GPT(mk('baseline')).num_params()/1e6:.2f}M   (layers={S['model'].n_layer}, d_model={S['model'].d_model}, vocab={S['model'].vocab_size}, ctx={S['model'].ctx})")

## 2. Find the biggest batch that fits (baseline)
Doubles the batch until the GPU runs out of memory, then bisects. Each trial is a real training
step (forward, backward, AdamW update), so optimizer memory is included.

In [ ]:
srch = ex.search_or_load("maxbatch_baseline", mk("baseline"), RESULTS, S)
B_FIXED = srch["max_batch"]
print("baseline max batch:", B_FIXED)

## 3. Train the baseline for 50M tokens at that batch size

In [ ]:
r0 = ex.run_or_load(mk("baseline"), ex.make_train_cfg(S, "01_baseline", B_FIXED), data, RESULTS)
B_FIXED = r0["batch_size"]          # (smaller only if a retry after OOM was needed)
ex.save(RESULTS, "fixed_batch", {"B_fixed": B_FIXED})
print(f"final val loss {r0['final_val_loss']:.4f} | {r0['tokens_per_sec_median']:,.0f} tokens/s | "
      f"peak {r0['peak_mem_allocated_mib']} MiB | {r0['wall_time_s']/60:.1f} min")

## 4. Report

In [ ]:
write_summary(RESULTS)
display(Image(f"{RESULTS}/loss_curves.png"))